# DeBERTa-v3-base: проверка на пяти фолдах и финальное обучение

Дообучаем DeBERTa для прогноза оценки от 1 до 5. Чтобы уложиться в 8 ГБ памяти Apple M1, обучаем **два верхних слоя энкодера, pooler и классификатор**; остальные веса заморожены. Используем одну эпоху и максимум 128 токенов.

Сначала оцениваем фиксированные настройки на пяти групповых фолдах, затем независимо обучаем финальную модель на всех очищенных данных. Каждый запуск начинает с исходных предобученных весов. Результаты финальной модели на обучающей выборке не считаем оценкой качества.

Циклы обучения и безопасное сохранение вынесены в `company_reviews/deberta_training.py`. Ноутбук использует тот же код, что и длительный запуск из терминала.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [2]:
from company_reviews.baseline import normalize_reviews_for_split
from company_reviews.deberta_training import (
    check_checkpoint,
    checkpoint_path,
    probability_median,
    run,
    training_config,
)
from company_reviews.training import read_training_data

seed = 2026
torch.set_num_threads(4)
device = "mps" if torch.backends.mps.is_available() else "cpu"
data_path = root / "data" / "raw" / "train.csv"
cache_dir = root / "data" / "cache"

/Users/vlanuka/Documents/ChatGPT/ncot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Данные и разбиение

Удаляем точные дубликаты отзывов, оставляя первую строку. Проверяем отсутствие пропусков и допустимость оценок. Совпадающие тексты с разными оценками требуют проверки и останавливают запуск.

Группы строим без учёта регистра и повторяющихся пробелов. Одна группа целиком попадает либо в обучение, либо в проверку. Разбиение `StratifiedGroupKFold` с пятью фолдами и `random_state=2026` совпадает с разбиением в `baselinev2.ipynb`. Каждый отзыв проверяется ровно один раз.

In [3]:
clean = read_training_data(data_path)
groups = normalize_reviews_for_split(clean["Review"])
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
folds = list(splitter.split(clean["Review"], clean["Rating"], groups))

for train_idx, valid_idx in folds:
    assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[valid_idx])

valid_rows = np.concatenate([valid_idx for _, valid_idx in folds])
assert np.array_equal(np.sort(valid_rows), np.arange(len(clean)))
print(f"Отзывов после удаления дубликатов: {len(clean):,}")
print(f"Групп: {groups.nunique():,}; устройство: {device}")

Отзывов после удаления дубликатов: 59,976
Групп: 59,975; устройство: mps


In [4]:
fold_sizes = pd.DataFrame(
    [
        {"фолд": number, "обучение": len(train_idx), "проверка": len(valid_idx)}
        for number, (train_idx, valid_idx) in enumerate(folds, start=1)
    ]
)
display(fold_sizes.set_index("фолд"))

,обучение,проверка
фолд,,
1,47980,11996
2,47980,11996
3,47981,11995
4,47981,11995
5,47982,11994


## Зафиксированные настройки

Используем исходный SentencePiece-токенизатор и закреплённую ревизию `microsoft/deberta-v3-base`. Дополняем вход до 128 токенов и обрезаем более длинные тексты. На ранее проверенном случайном образце 5 000 отзывов этот лимит обрезал около 13,7% текстов; это оценка по образцу.

Во время обучения оценки 1–5 преобразуются в классы 0–4. Функция потерь — cross-entropy. Оптимизатор AdamW получает разные скорости для энкодера и головы; первые 5% шагов отведены прогреву, затем скорость линейно снижается. Градиенты ограничены нормой 1. Настройки фиксированы до пятифолдовой оценки.

Перед каждым новым обучением заново устанавливаем seed и загружаем исходную модель. Это предотвращает перенос обученных весов между фолдами. Одинаковый seed повышает воспроизводимость; побитовое совпадение на разных устройствах и версиях библиотек не гарантируется.

In [5]:
configs = [
    training_config(data_path, clean, train_idx, valid_idx, device)
    for train_idx, valid_idx in folds
]
settings = configs[0]
parameter_names = [
    "model_id",
    "model_revision",
    "max_length",
    "batch_size",
    "num_epochs",
    "trainable_encoder_layers",
    "encoder_lr",
    "head_lr",
    "warmup_fraction",
    "max_grad_norm",
    "seed",
    "torch_version",
    "transformers_version",
]
display(pd.Series({name: settings[name] for name in parameter_names}, name="значение"))

model_id                                   microsoft/deberta-v3-base
model_revision              8ccc9b6f36199bec6961081d44eb72fb3f7353f3
max_length                                                       128
batch_size                                                         8
num_epochs                                                         1
trainable_encoder_layers                                           2
encoder_lr                                                   0.00003
head_lr                                                       0.0001
warmup_fraction                                                 0.05
max_grad_norm                                                    1.0
seed                                                            2026
torch_version                                                 2.14.1
transformers_version                                          4.57.6
Name: значение, dtype: object

## Обучение и проверка пяти фолдов

Следующая ячейка последовательно обучает недостающие фолды и получает проверочные вероятности. Ранее первый фолд занял 67,9 мин обучения и 13,7 мин проверки на Apple M1. Оставшиеся фолды запущены на другом компьютере с Apple M4 Max; время зависит от устройства и нагрузки.

Завершённая модель сохраняется вместе с токенизатором и параметрами. Каталог зависит от хешей данных и разбиения, ревизии модели, настроек и версий библиотек. Вероятности привязаны к хешу файла весов. При повторном запуске совместимые результаты переиспользуются. Прерванная эпоха запускается заново; уже завершённые фолды сохраняются.

Первый сохранённый фолд проверен отдельно: хеши исходного CSV, очищенных отзывов и разбиения совпали, проверочные вероятности соответствуют хешу весов. Его MAE равна 0,1472157 на 11 996 отзывах. Эта величина относится только к первому фолду.

Для длительного запуска из корня проекта можно использовать:

```bash
.venv/bin/python -u -m company_reviews.deberta_training --mode all --device mps
```

In [ ]:
progress_path = run(data_path, cache_dir, mode="cv", device=device)

## Качество

Из пяти вероятностей выбираем **медиану распределения**: первую оценку, для которой накопленная вероятность достигла 0,5. Такой прогноз минимизирует ожидаемую абсолютную ошибку для заданного распределения.

Показываем MAE каждого фолда, среднее и стандартное отклонение пяти значений. Общая MAE по всем проверочным прогнозам учитывает небольшую разницу в размерах фолдов. Это оценка фиксированных настроек, а не результат подбора гиперпараметров по этим же фолдам.

In [ ]:
results = json.loads(progress_path.read_text(encoding="utf-8"))
assert results["complete"] and len(results["folds"]) == 5

scores = pd.DataFrame(
    [
        {"фолд": item["fold"], "строк": item["rows"], "MAE": item["mae"]}
        for item in results["folds"]
    ]
)
display(scores.set_index("фолд").round(5))
print(f"Средняя MAE: {results['mean_mae']:.5f}")
print(f"Стандартное отклонение: {results['std_mae']:.5f}")
print(f"MAE всех проверочных прогнозов: {results['oof_mae']:.5f}")

In [ ]:
oof_predictions = np.zeros(len(clean), dtype=np.int64)
for (_, valid_idx), config, result in zip(
    folds, configs, results["folds"], strict=True
):
    model_dir = checkpoint_path(cache_dir, config)
    weights_digest = result["weights_sha256"][:12]
    probability_path = model_dir / f"valid_probabilities_{weights_digest}.npy"
    probabilities = np.load(probability_path, allow_pickle=False)
    oof_predictions[valid_idx] = probability_median(probabilities, len(valid_idx))

labels = clean["Rating"].to_numpy()
absolute_errors = np.abs(labels - oof_predictions)
assert np.isclose(absolute_errors.mean(), results["oof_mae"])

In [ ]:
by_rating = pd.DataFrame(
    [
        {
            "оценка": rating,
            "строк": int((labels == rating).sum()),
            "MAE": absolute_errors[labels == rating].mean(),
        }
        for rating in range(1, 6)
    ]
)
display(by_rating.set_index("оценка").round(5))

Общая MAE сильнее зависит от частых оценок 1 и 5. Таблица по отдельным оценкам помогает увидеть ошибки на редких классах 2–4. Для сравнения с baseline используем его пятифолдовый результат на таком же разбиении; одиночный фолд не подменяет среднее пяти фолдов.

## Финальная модель на всех данных

После оценки обучаем отдельную модель на всех 59 976 очищенных отзывах. Инициализация снова берётся из исходной предобученной DeBERTa, а не из одного из проверочных фолдов. Настройки остаются теми же. Одна эпоха на всех данных ожидаемо занимает около 85 минут на Apple M1.

Финальный каталог содержит веса, локальный токенизатор и `training_config.json` с отметкой `training_scope="full"` и числом обучающих строк. Этот каталог передаётся команде экспорта из README для подключения DeBERTa к API. Размер весов около 740 МБ; качество финальной модели оцениваем по проведённой кросс-валидации.

In [ ]:
run(data_path, cache_dir, mode="full", device=device)

full_config = training_config(
    data_path, clean, np.arange(len(clean)), np.array([], dtype=np.int64), device
)
final_model_dir = checkpoint_path(cache_dir, full_config)
assert check_checkpoint(final_model_dir, full_config)
print(f"Финальная модель: {final_model_dir}")
print(f"Обучающих строк: {full_config['train_rows']:,}")

## Ограничения

Проверены одна эпоха, длина 128 токенов и обучение двух верхних слоёв. Полное дообучение энкодера, более длинный вход и другие настройки этим экспериментом не оценивались. Групповое разбиение защищает от совпадающих нормализованных отзывов; семантически похожие тексты могут оставаться в разных группах.

Сохранение завершённого фолда позволяет продолжить длительный запуск после остановки. Промежуточные состояния оптимизатора внутри эпохи не сохраняются. Веса, вероятности и служебный файл прогресса находятся в `data/cache/` и не включаются в Git.